In [1]:
# Standard imports
import json
import logging
import os
from collections import defaultdict
from pathlib import Path

from dotenv import load_dotenv
from pydantic import SecretStr

# Configure logging
logging.basicConfig(level=logging.INFO)
log = logging.getLogger(__name__)

# Suppress httpx info logs
logging.getLogger("httpx").setLevel(logging.WARNING)

# Enable DEBUG logging for entity question gen to see invalid assertion details
logging.getLogger("benchmark_qed.autoq.question_gen.data_questions.entity_question_gen").setLevel(logging.DEBUG)

## 1. Load Local Questions from AP News v27

We'll use the candidate questions from data local question generation, which have:
- Named entities extracted
- Claims with source tracking
- Embeddings for similarity calculations

In [2]:
# Load environment variables
env_path = Path("../../local/ap_news_v2_27/.env")
load_dotenv(env_path)
print(f"Loaded environment from: {env_path}")

# Load candidate questions from AP News v27
data_path = Path("../../local/ap_news_v2_27/output/data_local_questions")
candidate_questions_path = data_path / "candidate_questions.json"

with open(candidate_questions_path) as f:
    raw_questions = json.load(f)

print(f"Loaded {len(raw_questions)} candidate questions")
print(f"\nSample question keys: {list(raw_questions[0].keys())}")

Loaded environment from: ..\..\local\ap_news_v2_27\.env
Loaded 349 candidate questions

Sample question keys: ['id', 'text', 'question_type', 'references', 'attributes']


In [3]:
# Check the attributes available on questions
sample_attrs = raw_questions[0].get("attributes", {})
print(f"Attributes keys: {list(sample_attrs.keys())}")
print(f"\nSample named_entities: {sample_attrs.get('named_entities', [])[:5]}")
print(f"\nNumber of claims: {len(sample_attrs.get('claims', []))}")

Attributes keys: ['input_question', 'period', 'location', 'named_entities', 'abstract_categories', 'background_information', 'reference_coverage', 'relevant_reference_count', 'reference_count', 'min_reference_similarity', 'max_reference_similarity', 'mean_reference_similarity', 'intra_inter_similarity_ratio', 'claim_count', 'claims', 'assertions', 'assertion_count', 'is_representative']

Sample named_entities: ['Alabama Supreme Court', 'wrongful death ruling on frozen embryos', 'wrongful death statute', 'frozen embryos', 'three couples']

Number of claims: 1


In [4]:
# Convert raw JSON to Question objects
from benchmark_qed.autoq.data_model.question import Question
from benchmark_qed.autoq.data_model.enums import QuestionType

local_questions = []
for raw in raw_questions:
    q = Question(
        id=raw["id"],
        text=raw["text"],
        question_type=QuestionType(raw["question_type"]),
        references=raw.get("references", []),
        attributes=raw.get("attributes", {}),
        embedding=raw.get("embedding"),
    )
    local_questions.append(q)

print(f"Converted {len(local_questions)} questions to Question objects")
print(f"Questions with embeddings: {sum(1 for q in local_questions if q.embedding is not None)}")
print(f"Questions with named_entities: {sum(1 for q in local_questions if q.attributes and q.attributes.get('named_entities'))}")
print(f"Questions with claims: {sum(1 for q in local_questions if q.attributes and q.attributes.get('claims'))}")

Converted 349 questions to Question objects
Questions with embeddings: 0
Questions with named_entities: 349
Questions with claims: 349


## 2. Analyze Entity Distribution

Let's see which entities appear across multiple questions - these are candidates for entity question generation.

In [5]:
# Group questions by entity
entity_to_questions: dict[str, list[Question]] = defaultdict(list)

for q in local_questions:
    if q.attributes and "named_entities" in q.attributes:
        for entity in q.attributes["named_entities"]:
            normalized = entity.strip().lower()
            if normalized:
                entity_to_questions[normalized].append(q)

print(f"Total unique entities: {len(entity_to_questions)}")

# Show distribution of entity frequency
freq_counts = defaultdict(int)
for entity, questions in entity_to_questions.items():
    freq_counts[len(questions)] += 1

print(f"\nEntity frequency distribution:")
for freq in sorted(freq_counts.keys(), reverse=True)[:10]:
    print(f"  {freq} questions: {freq_counts[freq]} entities")

Total unique entities: 1590

Entity frequency distribution:
  10 questions: 1 entities
  9 questions: 2 entities
  8 questions: 3 entities
  7 questions: 3 entities
  6 questions: 5 entities
  5 questions: 3 entities
  4 questions: 10 entities
  3 questions: 27 entities
  2 questions: 97 entities
  1 questions: 1439 entities


In [6]:
# Show top entities by question count
sorted_entities = sorted(
    entity_to_questions.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:20]

print("Top 20 entities by question count:")
for entity, questions in sorted_entities:
    print(f"  {entity}: {len(questions)} questions")

Top 20 entities by question count:
  associated press: 10 questions
  u.s. supreme court: 9 questions
  medicaid: 9 questions
  biden administration: 8 questions
  roe v. wade: 8 questions
  centers for disease control and prevention (cdc): 8 questions
  frozen embryos: 7 questions
  president joe biden: 7 questions
  in vitro fertilization (ivf): 7 questions
  alabama supreme court: 6 questions
  gender-affirming care: 6 questions
  florida: 6 questions
  u.s. food and drug administration (fda): 6 questions
  u.s. environmental protection agency (epa): 6 questions
  florida supreme court: 5 questions
  world health organization (who): 5 questions
  u.s. department of justice: 5 questions
  cancer diagnosis: 4 questions
  south korean government: 4 questions
  california: 4 questions


## 3. Setup LLM and Embedder

Configure the LLM for question generation and text embedder for similarity calculations.

In [7]:
# Setup LLM and Embedder (same pattern as autoq.ipynb)
import tiktoken
from benchmark_qed.autod.data_processor.embedding import TextEmbedder
from benchmark_qed.config.llm_config import LLMConfig
from benchmark_qed.config.llm_config import LLMProvider
from benchmark_qed.llm.factory import ModelFactory

# Model configs
API_KEY = SecretStr(os.getenv("OPENAI_API_KEY", ""))
EMBEDDING_MODEL = "text-embedding-3-large"
LLM_MODEL = "gpt-5.2"
LLM_PARAMS = {
    "temperature": 0.0,
    "seed": 42,
}
ENCODING_MODEL = "o200k_base"
CONCURRENT_REQUESTS = 32

# Create embedder and LLM
text_embedder = TextEmbedder(
    ModelFactory.create_embedding_model(
        LLMConfig(
            model=EMBEDDING_MODEL,
            api_key=API_KEY,
            llm_provider=LLMProvider.OpenAIEmbedding,
        )
    )
)
llm = ModelFactory.create_chat_model(
    model_config=LLMConfig(
        model=LLM_MODEL,
        api_key=API_KEY,
        llm_provider=LLMProvider.OpenAIChat,
        call_args=LLM_PARAMS,
    )
)
token_encoder = tiktoken.get_encoding(ENCODING_MODEL)

print(f"LLM model: {LLM_MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")

LLM model: gpt-5.2
Embedding model: text-embedding-3-large


## 4. Initialize Entity Question Generator

In [8]:
from benchmark_qed.autoq.question_gen.data_questions.entity_question_gen import (
    DataEntityQuestionGen,
)
from benchmark_qed.autoq.config import AssertionConfig

# Configure assertion generation
assertion_config = AssertionConfig()
assertion_config.entity.max_assertions = 20

# Create the generator (defaults to all 3 question types: bridge, comparison, intersection)
entity_gen = DataEntityQuestionGen(
    llm=llm,
    text_embedder=text_embedder,
    local_questions=local_questions,
    assertion_config=assertion_config,
    llm_params=LLM_PARAMS,
    # Entity grouping parameters
    min_questions_per_entity=2,  # Entity must appear in at least 2 local questions
    max_questions_per_entity=10,  # Cap local questions per entity for diversity
    # Quality filtering - all scores must be >= this threshold
    # Use 4 to get larger candidate pool, let sampler handle deduplication
    min_quality_score=4,
    concurrent_coroutines=CONCURRENT_REQUESTS,
    # question_types defaults to ["bridge", "comparison", "intersection"]
)

print("Entity question generator initialized")
print(f"Question types: {entity_gen.question_types}")

Entity question generator initialized
Question types: ['bridge', 'comparison', 'intersection']


## 5. Preview Entity Contexts

Before generating questions, let's preview what entity contexts will be created.

In [9]:
# Generate entity contexts (without LLM calls)
entity_contexts = entity_gen._generate_entity_contexts()

print(f"Generated {len(entity_contexts)} entity contexts")
print(f"\nSample contexts:")
for ctx in entity_contexts[:5]:
    print(f"\n  Entity: {ctx.entity}")
    print(f"  Local questions: {len(ctx.local_questions)}")
    print(f"  Claims: {len(ctx.claims)}")
    print(f"  Max questions to generate: {ctx.max_questions_to_generate}")

INFO:benchmark_qed.autoq.question_gen.data_questions.entity_question_gen:Found 151 entities with >= 2 questions (from 1590 total entities)


Generated 151 entity contexts

Sample contexts:

  Entity: alabama supreme court
  Local questions: 6
  Claims: 6
  Max questions to generate: 2

  Entity: frozen embryos
  Local questions: 7
  Claims: 8
  Max questions to generate: 2

  Entity: storage facility accident
  Local questions: 2
  Claims: 2
  Max questions to generate: 2

  Entity: extrauterine children
  Local questions: 2
  Claims: 2
  Max questions to generate: 2

  Entity: nebraska legislature
  Local questions: 2
  Claims: 3
  Max questions to generate: 2


In [10]:
# Preview claims for one entity context
if entity_contexts:
    ctx = entity_contexts[0]
    print(f"Entity: {ctx.entity}")
    print(f"\nLocal questions:")
    for q in ctx.local_questions:
        print(f"  - {q.text[:100]}...")
    
    print(f"\nClaims ({len(ctx.claims)} total):")
    for claim in ctx.claims[:5]:
        print(f"  [{claim['claim_id']}] {claim['statement'][:80]}...")

Entity: alabama supreme court

Local questions:
  - In the February 2024 Alabama Supreme Court wrongful death ruling about frozen embryos destroyed at a...
  - In the February 2024 Alabama Supreme Court IVF decision, what did the court say frozen embryos are c...
  - After the late-February 2024 Alabama Supreme Court IVF ruling, which Biden administration official t...
  - In March 2024 in Alabama, what liability does the new law remove for IVF providers for damage to or ...
  - In February 2024, after an Alabama Supreme Court ruling prompted IVF providers to pause services, wh...
  - In February 2024, what type of lawsuit did the Alabama Supreme Court say three Alabama couples could...

Claims (6 total):
  [c0] In the February 2024 Alabama Supreme Court wrongful death ruling allowing three ...
  [c1] In its Feb. 16, 2024 decision, the Republican-majority Alabama Supreme Court sai...
  [c2] After the late-February 2024 Alabama Supreme Court ruling that upended IVF treat...
  [c3] In Ma

In [11]:
# Preview the formatted prompt for one context
if entity_contexts:
    ctx = entity_contexts[0]
    claims_text = entity_gen._format_claims_for_prompt(ctx.claims, ctx.local_questions)
    print(f"Formatted claims for prompt:\n")
    print(claims_text[:2000])
    print("\n...")

Formatted claims for prompt:

Question: In the February 2024 Alabama Supreme Court wrongful death ruling about frozen embryos destroyed at a storage facility, what term did the justices use for the embryos?
Claims:
ID: c0
Statement: In the February 2024 Alabama Supreme Court wrongful death ruling allowing three couples to pursue claims after frozen embryos were destroyed at a storage facility, the justices referred to the embryos as their “extrauterine children.”


Question: In the February 2024 Alabama Supreme Court IVF decision, what did the court say frozen embryos are considered under Alabama state law?
Claims:
ID: c1
Statement: In its Feb. 16, 2024 decision, the Republican-majority Alabama Supreme Court said that frozen embryos created through IVF are considered children under Alabama state law.


Question: After the late-February 2024 Alabama Supreme Court IVF ruling, which Biden administration official traveled to Alabama to meet with patients and doctors?
Claims:
ID: c2
Stateme

## 6. Generate Entity Questions

Now let's generate the actual entity questions using the LLM.

In [12]:
# Generate entity questions
result = await entity_gen.agenerate(
    num_questions=50,  # Target 50 final questions
    oversample_factor=10.0,  # Generate 8x candidates
)

print(f"Generated {len(result.candidate_questions)} candidate questions")
print(f"Selected {len(result.selected_questions)} final questions")

INFO:benchmark_qed.autoq.question_gen.data_questions.entity_question_gen:Found 151 entities with >= 2 questions (from 1590 total entities)
INFO:benchmark_qed.autoq.question_gen.data_questions.entity_question_gen:Generated 151 entity contexts from 349 local questions (total 471 claims, ~191 max questions based on claim distribution)
INFO:benchmark_qed.autoq.question_gen.data_questions.entity_question_gen:Question types to generate: ['bridge', 'comparison', 'intersection']
INFO:benchmark_qed.autoq.question_gen.data_questions.entity_question_gen:Processing entity groups 0 to 32 of 151...
  0%|          | 0/32 [00:00<?, ?it/s]DEBUG:benchmark_qed.autoq.question_gen.data_questions.entity_question_gen:Generating bridge questions for entity 'fetal viability' (max 1)
DEBUG:benchmark_qed.autoq.question_gen.data_questions.entity_question_gen:Generating bridge questions for entity 'cancer diagnosis' (max 1)
DEBUG:benchmark_qed.autoq.question_gen.data_questions.entity_question_gen:Generating bridge

Generated 103 candidate questions
Selected 50 final questions


In [ ]:
# Save invalid assertions to file for review
output_dir = Path("../../local/ap_news_v2_27/output/data_entity_questions")
output_dir.mkdir(parents=True, exist_ok=True)

invalid_assertions = getattr(result, 'invalid_assertions', [])
print(f"Total invalid assertions: {len(invalid_assertions)}")

if invalid_assertions:
    invalid_assertions_path = output_dir / "invalid_assertions.json"
    with open(invalid_assertions_path, "w") as f:
        json.dump(invalid_assertions, f, indent=2)
    print(f"Saved invalid assertions to {invalid_assertions_path}")
    
    # Show summary by score pattern
    print("\nInvalid assertions by score pattern:")
    from collections import Counter
    patterns = Counter(
        f"grounding={a['grounding_score']}, relevance={a['relevance_score']}, verifiability={a['verifiability_score']}"
        for a in invalid_assertions
    )
    for pattern, count in patterns.most_common(10):
        print(f"  {pattern}: {count}")
else:
    print("No invalid assertions to save")

: 

In [ ]:
# Save filtered questions log for debugging
filter_stats = getattr(result, 'filter_stats', None)

if filter_stats:
    print(f"Filter stats summary:")
    print(f"  Total raw questions: {filter_stats.total_raw}")
    print(f"  Accepted: {filter_stats.accepted}")
    print(f"  Skipped - invalid claims: {filter_stats.skipped_invalid_claims}")
    print(f"  Skipped - few claims (<2): {filter_stats.skipped_few_claims}")
    print(f"  Skipped - low quality: {filter_stats.skipped_low_quality}")
    print(f"  Skipped - single document: {filter_stats.skipped_single_document}")
    print(f"  Skipped - parse error: {filter_stats.skipped_parse_error}")
    print(f"\nTotal filtered questions logged: {len(filter_stats.filtered_questions)}")
    
    if filter_stats.filtered_questions:
        filtered_path = output_dir / "filtered_questions.json"
        with open(filtered_path, "w") as f:
            json.dump(filter_stats.filtered_questions, f, indent=2)
        print(f"Saved filtered questions to {filtered_path}")
        
        # Show breakdown by filter reason
        from collections import Counter
        reason_counts = Counter(fq['reason'] for fq in filter_stats.filtered_questions)
        print("\nFiltered questions by reason:")
        for reason, count in reason_counts.most_common():
            print(f"  {reason}: {count}")
        
        # Show sample of filtered questions for each reason
        print("\n--- Sample filtered questions ---")
        for reason in reason_counts:
            samples = [fq for fq in filter_stats.filtered_questions if fq['reason'] == reason][:2]
            print(f"\n[{reason}] ({len([fq for fq in filter_stats.filtered_questions if fq['reason'] == reason])} total)")
            for fq in samples:
                print(f"  Type: {fq['question_type']}")
                print(f"  Text: {fq['text'][:150]}...")
                print(f"  Details: {fq['details']}")
                print(f"  Quality: {fq.get('quality', {})}")
                print()
else:
    print("No filter stats available (run with updated entity_question_gen.py)")

## 7. Analyze Generated Questions

In [ ]:
# Show quality score distribution
quality_scores = [
    q.attributes.get("quality_score", 0) 
    for q in result.selected_questions 
    if q.attributes
]

if quality_scores:
    print(f"Quality score statistics (selected questions):")
    print(f"  Min: {min(quality_scores):.3f}")
    print(f"  Max: {max(quality_scores):.3f}")
    print(f"  Mean: {sum(quality_scores) / len(quality_scores):.3f}")

In [ ]:
# Count total assertions across selected questions
total_assertions = 0
assertion_counts = []

for q in result.selected_questions:
    attrs = q.attributes or {}
    count = attrs.get("assertion_count", 0)
    assertion_counts.append(count)
    total_assertions += count

print(f"Total assertions across {len(result.selected_questions)} selected questions: {total_assertions}")
print(f"Average assertions per question: {total_assertions / len(result.selected_questions):.1f}" if result.selected_questions else "N/A")
print(f"Min assertions: {min(assertion_counts)}" if assertion_counts else "N/A")
print(f"Max assertions: {max(assertion_counts)}" if assertion_counts else "N/A")

# Distribution
from collections import Counter
count_dist = Counter(assertion_counts)
print(f"\nAssertion count distribution:")
for count in sorted(count_dist.keys()):
    print(f"  {count} assertions: {count_dist[count]} questions")

In [ ]:
# Show distribution of question types
from collections import Counter

type_counts = Counter(
    q.attributes.get("question_subtype", "unknown") 
    for q in result.selected_questions 
    if q.attributes
)

print("Question type distribution (selected):")
for q_type, count in type_counts.most_common():
    print(f"  {q_type}: {count} ({count/len(result.selected_questions)*100:.1f}%)")

# Also show candidate distribution
candidate_type_counts = Counter(
    q.attributes.get("question_subtype", "unknown") 
    for q in result.candidate_questions 
    if q.attributes
)

print("\nQuestion type distribution (candidates):")
for q_type, count in candidate_type_counts.most_common():
    print(f"  {q_type}: {count}")

## 8. Save Results

In [ ]:
# Save generated questions to JSON
output_dir = Path("../../local/ap_news_v2_27/output/data_entity_questions")
output_dir.mkdir(parents=True, exist_ok=True)

# Convert to JSON-serializable format (exclude embeddings to reduce file size)
def question_to_dict(q: Question) -> dict:
    return {
        "id": q.id,
        "text": q.text,
        "question_type": q.question_type.value,
        "references": q.references,
        "attributes": q.attributes,
    }

selected_data = [question_to_dict(q) for q in result.selected_questions]
candidate_data = [question_to_dict(q) for q in result.candidate_questions]

with open(output_dir / "selected_questions.json", "w") as f:
    json.dump(selected_data, f, indent=2)

with open(output_dir / "candidate_questions.json", "w") as f:
    json.dump(candidate_data, f, indent=2)

print(f"Saved {len(selected_data)} selected questions to {output_dir / 'selected_questions.json'}")
print(f"Saved {len(candidate_data)} candidate questions to {output_dir / 'candidate_questions.json'}")

In [ ]:
# Also save just the question text for easy review
text_data = [
    {
        "text": q.text,
        "type": q.attributes.get("question_subtype") if q.attributes else None,
        "entity": q.attributes.get("entity") if q.attributes else None,
    }
    for q in result.selected_questions
]

with open(output_dir / "selected_questions_text.json", "w") as f:
    json.dump(text_data, f, indent=2)

print(f"Saved question text to {output_dir / 'selected_questions_text.json'}")

In [ ]:
# Save assertions separately (same format as data_local and data_global questions)
questions_with_assertions = []

for q in result.selected_questions:
    attrs = q.attributes or {}
    assertions = attrs.get("assertions", [])
    
    if not assertions:
        continue
    
    # Sort by score descending and add rank
    sorted_assertions = sorted(assertions, key=lambda a: -a.get("score", 0))
    
    ranked_assertions = []
    for rank, assertion in enumerate(sorted_assertions, 1):
        sources = assertion.get("sources", [])
        ranked_assertion = {
            "statement": assertion["statement"],
            "source_count": len(sources),
            "score": assertion.get("score", 0),
            "reasoning": assertion.get("reasoning", ""),
            "rank": rank,
        }
        # Include validation if available
        assertion_attrs = assertion.get("attributes") or {}
        if "validation" in assertion_attrs:
            ranked_assertion["validation"] = assertion_attrs["validation"]
        ranked_assertions.append(ranked_assertion)
    
    questions_with_assertions.append({
        "question_id": q.id,
        "question_text": q.text,
        "assertions": ranked_assertions,
    })

# Save to file
with open(output_dir / "assertions.json", "w") as f:
    json.dump(questions_with_assertions, f, indent=4)

print(f"Saved assertions for {len(questions_with_assertions)} questions to {output_dir / 'assertions.json'}")